In [1]:
import pandas as pd

fake = pd.read_csv("Fake.csv")
real = pd.read_csv("True.csv")

fake["label"] = 1
real["label"] = 0

data = pd.concat([fake, real], ignore_index=True)
data = data[["text", "label"]]

data = data.dropna(subset=["text"])
data["text"] = data["text"].astype(str).str.strip()
data = data[data["text"] != ""]

print("Rows before removing duplicates:", len(data))
print("Duplicate texts:", data["text"].duplicated().sum())

data = data.drop_duplicates(subset=["text"]).reset_index(drop=True)

print("Rows after removing duplicates:", len(data))

data = data.sample(n=10000, random_state=42).reset_index(drop=True)

data.head()

Rows before removing duplicates: 44267
Duplicate texts: 5628
Rows after removing duplicates: 38639


,text,label
0,(This September 29 has been corrected to fix d...,0
1,WASHINGTON (Reuters) - The Secret Service is i...,0
2,PRISTINA (Reuters) - Kosovo s center-right coa...,0
3,WASHINGTON (Reuters) - Senator Richard Blument...,0
4,"SHARM AL-SHEIKH, Egypt (Reuters) - Egyptian Pr...",0


In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    data["text"],
    data["label"],
    test_size=0.2,
    random_state=42,
    stratify=data["label"]
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Train label distribution:\n", y_train.value_counts())
print("Test label distribution:\n", y_test.value_counts())

Train size: 8000
Test size: 2000
Train label distribution:
 label
0    4398
1    3602
Name: count, dtype: int64
Test label distribution:
 label
0    1100
1     900
Name: count, dtype: int64


## Baseline Model

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train_tfidf, y_train)

baseline_preds = baseline_model.predict(X_test_tfidf)

baseline_acc = accuracy_score(y_test, baseline_preds)

print("Baseline Accuracy:", baseline_acc)

Baseline Accuracy: 0.9785


## DistilBERT

In [4]:
from transformers import DistilBertTokenizer
from torch.utils.data import Dataset
import torch

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = NewsDataset(X_train, y_train, tokenizer)
test_dataset = NewsDataset(X_test, y_test, tokenizer)

C:\Users\cbram\anaconda3\envs\deep-learning\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import torch
from torch.utils.data import DataLoader
from transformers import DistilBertForSequenceClassification
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)
model.to(device)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

model.train()

for epoch in range(1):
    total_loss = 0

    for batch in tqdm(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print("Epoch", epoch + 1, "Loss:", total_loss / len(train_loader))

print("Training done")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda


100%|██████████| 500/500 [03:54<00:00,  2.13it/s]

Epoch 1 Loss: 0.025053250671888235
Training done


In [6]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels, all_preds, average="binary"
)

print("Accuracy:", acc)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds))

Accuracy: 0.998
Precision: 0.995575221238938
Recall: 1.0
F1-score: 0.9977827050997783

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1100
           1       1.00      1.00      1.00       900

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



In [7]:
train_loader_eval = DataLoader(train_dataset, batch_size=16, shuffle=False)

model.eval()

train_preds = []
train_labels = []

with torch.no_grad():
    for batch in train_loader_eval:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)

        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(batch["labels"].cpu().numpy())

train_acc = accuracy_score(train_labels, train_preds)
print("Train Accuracy:", train_acc)

Train Accuracy: 0.998375


In [8]:
import torch.nn.functional as F

def predict_news(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred].item()

    label_map = {0: "REAL", 1: "FAKE"}
    return label_map[pred], confidence

In [9]:
def get_fake_score(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)

    return probs[0][1].item()

In [10]:
import re

def perturbation_explanation(text, top_k=10):
    words = re.findall(r"\b\w+\b", text)
    clean_text = " ".join(words)
    original_score = get_fake_score(clean_text)

    word_importance = []

    for i in range(len(words)):
        new_text = " ".join(words[:i] + words[i+1:])
        new_score = get_fake_score(new_text)
        importance = original_score - new_score
        word_importance.append((words[i], importance))

    word_importance = sorted(word_importance, key=lambda x: x[1], reverse=True)
    return word_importance[:top_k]

In [11]:
def explain_words(words):
    explanations = []

    for word, score in words:
        w = word.lower()

        if w in ["breaking", "shocking", "unbelievable"]:
            reason = "This word suggests sensational or exaggerated language, often seen in fake news."
        
        elif w in ["washington", "reuters", "senator"]:
            reason = "This word refers to a real entity or location, which is more common in credible news."
        
        elif w in ["thank", "god", "i", "we"]:
            reason = "This reflects personal or emotional tone, which is less common in formal news."
        
        else:
            reason = "This word influenced the model's decision based on learned patterns."

        explanations.append((word, score, reason))

    return explanations

In [19]:
sample_text = X_test.iloc[0]

top_words = perturbation_explanation(sample_text, top_k=12)

stop_words = {
    "the","a","an","and","or","but","is","are","was","were",
    "to","of","in","on","for","with","at","by","all","who",
    "this","s","no"
}

filtered_top_words = [
    (w, s) for w, s in top_words 
    if w.lower() not in stop_words and len(w) > 2
]

word_explanations = explain_words(filtered_top_words)

explanation_df = pd.DataFrame(word_explanations, columns=["word", "importance", "reason"])
explanation_df

,word,importance,reason
0,just,0.000010,This word influenced the model's decision base...
1,you,0.000008,This word influenced the model's decision base...
2,wanted,0.000006,This word influenced the model's decision base...


In [20]:
explanation_df = pd.DataFrame(filtered_top_words, columns=["word", "importance"])
explanation_df

,word,importance
0,just,0.000010
1,you,0.000008
2,wanted,0.000006


In [21]:
label, confidence = predict_news(sample_text)

print("\nPrediction:", label)
print("Confidence:", confidence)

print("\nOverall Explanation:")

if label == "FAKE":
    print("The model predicts this news as FAKE because it contains emotional, sensational, or informal language patterns.")
else:
    print("The model predicts this news as REAL because it contains structured, factual, and entity-based language.")


Prediction: FAKE
Confidence: 0.9993622899055481

Overall Explanation:
The model predicts this news as FAKE because it contains emotional, sensational, or informal language patterns.


In [22]:
comparison_df = pd.DataFrame({
    "Model": ["Logistic Regression", "DistilBERT"],
    "Accuracy": [baseline_acc, acc]
})

comparison_df

,Model,Accuracy
0,Logistic Regression,0.9785
1,DistilBERT,0.9980


In [23]:
print("MODEL ANALYSIS ")
print("Baseline Accuracy:", baseline_acc)
print("DistilBERT Accuracy:", acc)
print("Train Accuracy:", train_acc)

print("\nObservation:")
print("Very high accuracy suggests the dataset may contain strong patterns or shortcuts.")
print("The model may be learning source/style differences instead of actual misinformation reasoning.")

MODEL ANALYSIS 
Baseline Accuracy: 0.9785
DistilBERT Accuracy: 0.998
Train Accuracy: 0.998375

Observation:
Very high accuracy suggests the dataset may contain strong patterns or shortcuts.
The model may be learning source/style differences instead of actual misinformation reasoning.


In [24]:
for i in range(3):
    text = X_test.iloc[i]
    print("\nTEXT:", text[:200])
    print("\nTop words:")
    
    top_words = perturbation_explanation(text, top_k=5)
    
    for word, score in top_words:
        print(word, score)


TEXT: MI, PA, OH and NY residents were all warned he could be anywhere in their state. Thank God he was caught and is no longer a danger to innocent people who may have encountered him Kudos to the PA State

Top words:
OH 3.55839729309082e-05
just 1.0371208190917969e-05
and 7.748603820800781e-06
you 7.510185241699219e-06
I 7.510185241699219e-06

TEXT: Thomas  article can be read below:My dad called Martin Luther King an  agitator.  I bet a lot of white dads did. Seems like every time Dr. King showed up somewhere, things got torn up or burned up.So 

Top words:
poke 5.543231964111328e-05
of 1.4066696166992188e-05
killed 1.239776611328125e-05
squared 1.0967254638671875e-05
Thomas 9.417533874511719e-06

TEXT: WASHINGTON (Reuters) - A secret meeting at a neighbor’s house, a drive through a back country road, and a clandestine flight on a military jet marked Judge Neil Gorsuch’s journey to the White House th

Top words:
A 0.009785592555999756
s 0.00497204065322876
a 0.0018100738525390625
s

In [25]:
model.save_pretrained("saved_model")
tokenizer.save_pretrained("saved_model")
print("Model and tokenizer saved.")

Model and tokenizer saved.
